# ✈️ Flight Aircraft Usage Dashboard
## Interactive Analysis of Commercial Aircraft Operations

**Project:** DATA 400 Final Project - Fall 2025  
**Authors:** Kevin Tran & Will Barlett
**Institution:** Dickinson College  
**Data Source:** Amadeus API  

---

### 📋 **Notebook Overview**

This notebook creates an interactive dashboard to analyze aircraft usage patterns across major US airlines. The dashboard provides:
- Real-time filtering by airline, route, aircraft type, and flight characteristics
- Interactive pie chart visualizations
- Live currency conversion (EUR/USD)
- Comprehensive aircraft family classification

**Data Period:** November 10 - December 19, 2025  
**Total Records:** ~800,000+ flight observations


---
## 📦 1. Setup & Imports

Import required libraries for data processing, visualization, and dashboard creation.

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dropbox
import os
import io
from dropbox.exceptions import ApiError
from scipy import stats
import requests
from datetime import datetime
import dash
from dash import dcc, html, Input, Output, State
import plotly.graph_objects as go
import base64

print("All libraries imported.")

All libraries imported.


---
## 💱 2. Currency Exchange Rate Setup

Fetch live EUR to USD exchange rate for real-time price conversion in the dashboard.


In [2]:
def get_live_exchange_rate():
    """Fetch live EUR to USD rate, fallback to default if API fails"""
    try:
        response = requests.get('https://api.exchangerate-api.com/v4/latest/EUR', timeout=5)
        if response.status_code == 200:
            data = response.json()
            return data['rates']['USD'], datetime.now().strftime('%b %d, %I:%M %p')
    except:
        pass
    return 1.05, "Offline"

# Get rate when app starts
EUR_TO_USD_RATE, RATE_UPDATE_TIME = get_live_exchange_rate()
print(f"Exchange rate loaded: 1 EUR = ${EUR_TO_USD_RATE:.3f} USD (Updated: {RATE_UPDATE_TIME})")

Exchange rate loaded: 1 EUR = $1.170 USD (Updated: Dec 19, 08:11 AM)


---
## 📥 3. Data Loading from Dropbox

Connect to Dropbox API and load all flight data CSV files. The data is combined into a single DataFrame for analysis.


In [3]:
# Save dropbox credentials
DROPBOX_APP_KEY = "9gghuhtkyrtyn2u"
DROPBOX_APP_SECRET = "tqtvwbr15gbwseg"
DROPBOX_REFRESH_TOKEN = "LpMKV__by1wAAAAAAAAAAYn_oi2fIW0SBBqF0n6gap1WavDnJGFRMcVWQol7X5Vy"

# Create Dropbox client
dbx = dropbox.Dropbox(
    app_key=DROPBOX_APP_KEY,
    app_secret=DROPBOX_APP_SECRET,
    oauth2_refresh_token=DROPBOX_REFRESH_TOKEN
)

print("Reading files from Dropbox...")

# List all files
result = dbx.files_list_folder('/flight_data')

dfs = []
for entry in result.entries:
    if entry.name.endswith('.csv'):
        print(f"Reading: {entry.name}")
        
        # Download file content
        metadata, response = dbx.files_download(entry.path_lower)
        
        # Read directly into pandas
        df = pd.read_csv(io.BytesIO(response.content))
        dfs.append(df)

# Combine
combined_df = pd.concat(dfs, ignore_index=True)

# Process
combined_df['route'] = combined_df['origin'] + ' → ' + combined_df['destination']
combined_df['time_collected'] = pd.to_datetime(combined_df['time_collected'])
combined_df['departure_date'] = pd.to_datetime(combined_df['departure_date'])

# Save locally
combined_df.to_csv('combined_flight_data.csv', index=False)

# Load combined data
df = pd.read_csv("combined_flight_data.csv")

# Convert to datetime with error handling
try:
    if df['departure_date'].dtype != 'datetime64[ns]':
        df['departure_date'] = pd.to_datetime(df['departure_date'], errors='coerce')
except Exception as e:
    print(f"Error with departure_date: {e}")
    # Try alternative parsing
    df['departure_date'] = pd.to_datetime(df['departure_date'].astype(str), errors='coerce')

try:
    if df['departure_time'].dtype != 'datetime64[ns]':
        df['departure_time'] = pd.to_datetime(df['departure_time'], errors='coerce')
except Exception as e:
    print(f"Error with departure_time: {e}")
    df['departure_time'] = pd.to_datetime(df['departure_time'].astype(str), errors='coerce')

try:
    if df['time_collected'].dtype != 'datetime64[ns]':
        df['time_collected'] = pd.to_datetime(df['time_collected'], errors='coerce')
except Exception as e:
    print(f"Error with time_collected: {e}")
    df['time_collected'] = pd.to_datetime(df['time_collected'].astype(str), errors='coerce')

# Round departure time to nearest 15 minutes (flights might vary by a few minutes)
df['departure_time_rounded'] = df['departure_time'].dt.floor('15min')

# Create flight ID for each unique flight
df['flight_id'] = (
    df['origin'].astype(str) + '_' + 
    df['destination'].astype(str) + '_' + 
    df['departure_date'].dt.strftime('%Y%m%d') + '_' + 
    df['departure_time_rounded'].dt.strftime('%H%M') + '_' + 
    df['airline'].astype(str)
)
df.head()

Reading files from Dropbox...
Reading: flight_data_20251110_210135.csv
Reading: flight_data_20251111_031726.csv
Reading: flight_data_20251111_130430.csv
Reading: flight_data_20251112_021940.csv
Reading: flight_data_20251112_130852.csv
Reading: flight_data_20251113_022155.csv
Reading: flight_data_20251113_130822.csv
Reading: flight_data_20251114_022042.csv
Reading: flight_data_20251114_130600.csv
Reading: flight_data_20251115_021721.csv
Reading: flight_data_20251115_125944.csv
Reading: flight_data_20251116_022726.csv
Reading: flight_data_20251116_125744.csv
Reading: flight_data_20251117_022452.csv
Reading: flight_data_20251117_132325.csv
Reading: flight_data_20251118_025158.csv
Reading: flight_data_20251118_130553.csv
Reading: flight_data_20251119_021933.csv
Reading: flight_data_20251119_130946.csv
Reading: flight_data_20251120_021830.csv
Reading: flight_data_20251120_130617.csv
Reading: flight_data_20251121_022030.csv
Reading: flight_data_20251121_130024.csv
Reading: flight_data_202511

,time_collected,origin,destination,departure_date,days_until_departure,price,currency,airline,number_of_stops,departure_time,arrival_time,total_duration,aircraft_type,cabin_class,bookable_seats,route,departure_time_rounded,flight_id
0,2025-11-10 20:41:58,JFK,LAX,2025-11-11,1,144.27,EUR,F9,1,2025-11-11 06:25:00,2025-11-11T12:25:00,PT9H,"32Q,32Q",ECONOMY,4,JFK → LAX,2025-11-11 06:15:00,JFK_LAX_20251111_0615_F9
1,2025-11-10 20:41:58,JFK,LAX,2025-11-11,1,149.13,EUR,F9,1,2025-11-11 07:59:00,2025-11-11T19:53:00,PT14H54M,"32Q,32Q",ECONOMY,4,JFK → LAX,2025-11-11 07:45:00,JFK_LAX_20251111_0745_F9
2,2025-11-10 20:41:58,JFK,LAX,2025-11-11,1,149.13,EUR,F9,1,2025-11-11 07:59:00,2025-11-11T20:42:00,PT15H43M,"32Q,32N",ECONOMY,4,JFK → LAX,2025-11-11 07:45:00,JFK_LAX_20251111_0745_F9
3,2025-11-10 20:41:58,JFK,LAX,2025-11-11,1,210.02,EUR,F9,0,2025-11-11 11:29:00,2025-11-11T14:32:00,PT6H3M,32Q,ECONOMY,4,JFK → LAX,2025-11-11 11:15:00,JFK_LAX_20251111_1115_F9
4,2025-11-10 20:41:58,JFK,LAX,2025-11-11,1,256.07,EUR,AS,1,2025-11-11 07:00:00,2025-11-11T15:17:00,PT11H17M,"73H,73J",ECONOMY,7,JFK → LAX,2025-11-11 07:00:00,JFK_LAX_20251111_0700_AS


---
## 💾 4. Data Export

Save the combined dataset locally for backup and future analysis.


In [4]:
df.to_csv('Nov10_Dec19.csv', index=False)

---
## 🏢 5. Airline Mapping Configuration

Map airline codes to full airline names for better readability in the dashboard.

**Supported Airlines:**
- **AS** - Alaska Airlines
- **B6** - JetBlue Airways
- **F9** - Frontier Airlines
- **HA** - Hawaiian Airlines
- **UA** - United Airlines


In [5]:
# Airline code to name mapping
airline_mapping = {
    'AS': 'Alaska Airlines',
    'B6': 'JetBlue Airways',
    'F9': 'Frontier Airlines',
    'HA': 'Hawaiian Airlines',
    'UA': 'United Airlines'
}

# Apply the mapping to create a new column with full airline names
df['airline_name'] = df['airline'].map(airline_mapping)

# Check the results
print(df[['airline', 'airline_name']].head())

  airline       airline_name
0      F9  Frontier Airlines
1      F9  Frontier Airlines
2      F9  Frontier Airlines
3      F9  Frontier Airlines
4      AS    Alaska Airlines


---
## ✈️ 6. Aircraft Classification System

### Aircraft Code Mapping
Translate IATA aircraft codes to full aircraft names and organize them into families and brands.

### Classification Hierarchy:
1. **Aircraft Code** (e.g., "73J") → **Aircraft Name** (e.g., "Boeing 737-900ER")
2. **Aircraft Name** → **Aircraft Family** (e.g., "Boeing 737 NG")
3. **Aircraft Family** → **Brand/Manufacturer** (e.g., "Boeing")


In [6]:
import pandas as pd
from collections import Counter

# Aircraft code to name mapping
aircraft_mapping = {
    '321': 'Airbus A321',
    '320': 'Airbus A320',
    '319': 'Airbus A319',
    '32Q': 'Airbus A321neo',
    '32S': 'Airbus A32S (A320 Family Generic)',
    '32N': 'Airbus A320neo',
    '737': 'Boeing 737-700',
    '738': 'Boeing 737-800',
    '739': 'Boeing 737-900',
    '7M8': 'Boeing 737 MAX 8',
    '7M9': 'Boeing 737 MAX 9',
    '73G': 'Boeing 737-700',
    '73W': 'Boeing 737-700 (Winglets)',
    '73J': 'Boeing 737-900ER',
    '73H': 'Boeing 737-800',
    '752': 'Boeing 757-200',
    '753': 'Boeing 757-300',
    '777': 'Boeing 777-200',
    '789': 'Boeing 787-9 Dreamliner',
    '781': 'Boeing 787-10 Dreamliner',
    'E70': 'Embraer E170',
    'E7W': 'Embraer E175',
    'E75': 'Embraer E175',
    'CRJ': 'Bombardier CRJ (Generic)',  # Updated to Bombardier
    'CR5': 'Bombardier CRJ500',
    'CR7': 'Bombardier CRJ700',  # DUPLICATE REMOVED - kept the newer Bombardier version
    'CR9': 'Bombardier CRJ900',
    'ER4': 'Embraer ERJ-145',
    'ERJ': 'Embraer Regional Jet (Generic)',
    '223': 'Bombardier CS300 / Airbus A220-300',
    '332': 'Airbus A330-200 (Wide-body)',
    '330': 'Airbus A330 (Generic)',
    '788': 'Boeing 787-8 Dreamliner',
    '77W': 'Boeing 777-300ER',
}



def count_aircraft(dataframe, airlines=None, routes=None):
    """
    Count aircraft segments from flights, optionally filtered by airline(s) and/or route(s)
    Returns a DataFrame with aircraft names, counts, and filter identifiers
    
    Parameters:
    - airlines: single airline name (str) or list of airline names
    - routes: single route (str) or list of routes
    """
    # Convert single values to lists for consistent handling
    if airlines is not None and isinstance(airlines, str):
        airlines = [airlines]
    if routes is not None and isinstance(routes, str):
        routes = [routes]
    
    # Filter dataframe
    filtered_df = dataframe.copy()
    if airlines:
        filtered_df = filtered_df[filtered_df['airline_name'].isin(airlines)]
    if routes:
        filtered_df = filtered_df[filtered_df['route'].isin(routes)]
    
    # Count all aircraft segments
    aircraft_counts = Counter()
    
    for aircraft_str in filtered_df['aircraft_type']:
        # Split by comma and clean
        codes = [code.strip() for code in str(aircraft_str).split(',')]
        # Map to names and count
        for code in codes:
            aircraft_name = aircraft_mapping.get(code, code)
            aircraft_counts[aircraft_name] += 1
    
    # Convert to DataFrame
    aircraft_df = pd.DataFrame(aircraft_counts.items(), columns=['Aircraft', 'Count'])
    aircraft_df = aircraft_df.sort_values('Count', ascending=False).reset_index(drop=True)
    
    # Add identifier columns
    if airlines:
        aircraft_df['Airlines'] = ', '.join(airlines)
    else:
        aircraft_df['Airlines'] = 'All'
    
    if routes:
        aircraft_df['Routes'] = ', '.join(routes)
    else:
        aircraft_df['Routes'] = 'All'
    
    return aircraft_df
# Example usage:

# All data - will show all actual airlines and routes
result1 = count_aircraft(df)
print(result1.head())

# Single airline - will show that airline and all routes it operates
result2 = count_aircraft(df, airlines='Alaska Airlines')
print(result2.head())

# Single route - will show all airlines on that route
result3 = count_aircraft(df, routes='SEA → BOS')
print(result3.head())

# Multiple specific filters
result4 = count_aircraft(df, 
                        airlines=['Alaska Airlines', 'United Airlines'],
                        routes=['SEA → BOS', 'LAX → JFK'])
print(result4.head())

                            Aircraft   Count Airlines Routes
0                   Boeing 737-900ER  229140      All    All
1                     Boeing 737-700  141162      All    All
2                     Airbus A321neo  133412      All    All
3  Airbus A32S (A320 Family Generic)  126331      All    All
4                     Boeing 737-900  122211      All    All
           Aircraft   Count         Airlines Routes
0  Boeing 737-900ER  228885  Alaska Airlines    All
1      Embraer E175   40921  Alaska Airlines    All
2  Boeing 737 MAX 9   11109  Alaska Airlines    All
3    Boeing 737-800    7384  Alaska Airlines    All
4       Airbus A319    3138  Alaska Airlines    All
           Aircraft  Count Airlines     Routes
0  Boeing 737-900ER  13369      All  SEA → BOS
1    Boeing 737-900  10453      All  SEA → BOS
2  Boeing 737 MAX 9  10271      All  SEA → BOS
3    Boeing 737-800   7025      All  SEA → BOS
4      Embraer E175   6967      All  SEA → BOS
           Aircraft  Count              

In [7]:
# Aircraft family mapping - COMPLETE
aircraft_family_mapping = {
    # Airbus A320 Family (CEO - Current Engine Option)
    'Airbus A321': 'Airbus A320ceo Family',
    'Airbus A320': 'Airbus A320ceo Family',
    'Airbus A319': 'Airbus A320ceo Family',
    'Airbus A32S (A320 Family Generic)': 'Airbus A320ceo Family',
    
    # Airbus A320 Family (NEO - New Engine Option)
    'Airbus A321neo': 'Airbus A320neo Family',
    'Airbus A320neo': 'Airbus A320neo Family',
    
    # Airbus A330
    'Airbus A330-200 (Wide-body)': 'Airbus A330',
    'Airbus A330 (Generic)': 'Airbus A330',
    
    # Airbus A220 (formerly Bombardier C Series)
    'Bombardier CS300 / Airbus A220-300': 'Airbus A220',
    
    # Boeing 737 Classic/NG
    'Boeing 737-700': 'Boeing 737 NG',
    'Boeing 737-700 (Winglets)': 'Boeing 737 NG',
    'Boeing 737-800': 'Boeing 737 NG',
    'Boeing 737-900': 'Boeing 737 NG',
    'Boeing 737-900ER': 'Boeing 737 NG',
    
    # Boeing 737 MAX
    'Boeing 737 MAX 8': 'Boeing 737 MAX',
    'Boeing 737 MAX 9': 'Boeing 737 MAX',
    
    # Boeing 757
    'Boeing 757-200': 'Boeing 757',
    'Boeing 757-300': 'Boeing 757',
    
    # Boeing 777
    'Boeing 777-200': 'Boeing 777',
    'Boeing 777-300ER': 'Boeing 777',
    
    # Boeing 787 Dreamliner
    'Boeing 787-8 Dreamliner': 'Boeing 787 Dreamliner',
    'Boeing 787-9 Dreamliner': 'Boeing 787 Dreamliner',
    'Boeing 787-10 Dreamliner': 'Boeing 787 Dreamliner',
    
    # Embraer E-Jets
    'Embraer E170': 'Embraer E-Jet',
    'Embraer E175': 'Embraer E-Jet',
    'Embraer ERJ-145': 'Embraer ERJ',
    'Embraer Regional Jet (Generic)': 'Embraer ERJ',
    
    # Bombardier CRJ
    'Bombardier CRJ500': 'Bombardier CRJ',
    'Bombardier CRJ700': 'Bombardier CRJ',
    'Bombardier CRJ900': 'Bombardier CRJ',
    'Canadair Regional Jet (Generic)': 'Bombardier CRJ',
    'Canadair Regional Jet 500': 'Bombardier CRJ',
    'Canadair Regional Jet 700': 'Bombardier CRJ',
    'Canadair Regional Jet 900': 'Bombardier CRJ',
}

# Custom sort order for aircraft - COMPLETE
aircraft_sort_order = {
    # Airbus family - sorted by size and generation
    'Airbus A220': 1,
    'Bombardier CS300 / Airbus A220-300': 2,
    'Airbus A319': 3,
    'Airbus A320': 4,
    'Airbus A320neo': 5,
    'Airbus A321': 6,
    'Airbus A321neo': 7,
    'Airbus A32S (A320 Family Generic)': 8,
    'Airbus A330 (Generic)': 9,
    'Airbus A330-200 (Wide-body)': 10,
    
    # Boeing family - sorted by model number and generation
    'Boeing 737-700': 20,
    'Boeing 737-700 (Winglets)': 21,
    'Boeing 737-800': 22,
    'Boeing 737-900': 23,
    'Boeing 737-900ER': 24,
    'Boeing 737 MAX 8': 25,
    'Boeing 737 MAX 9': 26,
    'Boeing 757-200': 27,
    'Boeing 757-300': 28,
    'Boeing 777-200': 29,
    'Boeing 777-300ER': 30,
    'Boeing 787-8 Dreamliner': 31,
    'Boeing 787-9 Dreamliner': 32,
    'Boeing 787-10 Dreamliner': 33,
    
    # Bombardier/Canadair
    'Bombardier CRJ500': 40,
    'Canadair Regional Jet 500': 41,
    'Bombardier CRJ700': 42,
    'Canadair Regional Jet 700': 43,
    'Canadair Regional Jet (Generic)': 44,
    'Bombardier CRJ900': 45,
    'Canadair Regional Jet 900': 46,
    
    # Embraer
    'Embraer ERJ-145': 50,
    'Embraer Regional Jet (Generic)': 51,
    'Embraer E170': 52,
    'Embraer E175': 53,
}

# Aircraft brand/manufacturer mapping
aircraft_brand_mapping = {
    # Airbus
    'Airbus A220': 'Airbus',
    'Airbus A319': 'Airbus',
    'Airbus A320': 'Airbus',
    'Airbus A320neo': 'Airbus',
    'Airbus A321': 'Airbus',
    'Airbus A321neo': 'Airbus',
    'Airbus A32S (A320 Family Generic)': 'Airbus',
    'Airbus A330 (Generic)': 'Airbus',
    'Airbus A330-200 (Wide-body)': 'Airbus',
    'Bombardier CS300 / Airbus A220-300': 'Airbus',
    
    # Boeing
    'Boeing 737-700': 'Boeing',
    'Boeing 737-700 (Winglets)': 'Boeing',
    'Boeing 737-800': 'Boeing',
    'Boeing 737-900': 'Boeing',
    'Boeing 737-900ER': 'Boeing',
    'Boeing 737 MAX 8': 'Boeing',
    'Boeing 737 MAX 9': 'Boeing',
    'Boeing 757-200': 'Boeing',
    'Boeing 757-300': 'Boeing',
    'Boeing 777-200': 'Boeing',
    'Boeing 777-300ER': 'Boeing',
    'Boeing 787-8 Dreamliner': 'Boeing',
    'Boeing 787-9 Dreamliner': 'Boeing',
    'Boeing 787-10 Dreamliner': 'Boeing',
    
    # Bombardier
    'Bombardier CRJ500': 'Bombardier',
    'Bombardier CRJ700': 'Bombardier',
    'Bombardier CRJ900': 'Bombardier',
    'Canadair Regional Jet (Generic)': 'Bombardier',
    'Canadair Regional Jet 500': 'Bombardier',
    'Canadair Regional Jet 700': 'Bombardier',
    'Canadair Regional Jet 900': 'Bombardier',
    
    # Embraer
    'Embraer E170': 'Embraer',
    'Embraer E175': 'Embraer',
    'Embraer ERJ-145': 'Embraer',
    'Embraer Regional Jet (Generic)': 'Embraer',
}

# Airline brand colors (primary livery colors)
airline_colors = {
    'Alaska Airlines': '#01426A',      # Alaska Blue
    'JetBlue Airways': '#0033A0',      # JetBlue Blue  
    'Frontier Airlines': '#6CC24A',    # Frontier Green
    'Hawaiian Airlines': '#6F2C91',    # Hawaiian Purple
    'United Airlines': '#0033A0',      # United Blue
}

# Airline logo file paths (relative to notebook)
airline_logo_paths = {
    'Alaska Airlines': 'assets/airline_logos/alaska_airlines.svg',
    'JetBlue Airways': 'assets/airline_logos/jetblue_airways.svg',
    'Frontier Airlines': 'assets/airline_logos/frontier_airlines.svg',
    'Hawaiian Airlines': 'assets/airline_logos/hawaiian_airlines.svg',
    'United Airlines': 'assets/airline_logos/united_airlines.svg',
}

---
## 🔧 7. Data Processing Functions

Define helper functions for:
- Duration parsing (ISO 8601 format to minutes)
- Aircraft counting with multi-dimensional filtering
- Dynamic aggregation by airline, route, aircraft family, and more


In [8]:
def parse_duration_to_minutes(duration_str):
    """Convert ISO 8601 duration to minutes"""
    try:
        if pd.isna(duration_str):
            return None
        duration_str = str(duration_str)
        # Handle PT format (e.g., PT6H3M, PT9H)
        if duration_str.startswith('PT'):
            hours = 0
            minutes = 0
            duration_str = duration_str[2:]  # Remove PT
            
            if 'H' in duration_str:
                parts = duration_str.split('H')
                hours = int(parts[0])
                duration_str = parts[1] if len(parts) > 1 else ''
            
            if 'M' in duration_str and duration_str:
                minutes = int(duration_str.replace('M', ''))
            
            return hours * 60 + minutes
        else:
            # Try parsing as timedelta
            td = pd.to_timedelta(duration_str)
            return td.total_seconds() / 60
    except:
        return None

# Replace the count_aircraft function with this corrected version
def count_aircraft(dataframe, airlines=None, routes=None, aircraft_families=None, origins=None, destinations=None, brands=None,
                  stops=None, cabins=None, redeye_only=False, days_until_range=None, 
                  departure_hour_range=None, price_range=None, duration_minutes_range=None, seats_range=None):
    """
    Count aircraft segments from flights, optionally filtered by multiple criteria
    """
    # Convert single values to lists for consistent handling
    if airlines is not None and isinstance(airlines, str):
        airlines = [airlines]
    if routes is not None and isinstance(routes, str):
        routes = [routes]
    if aircraft_families is not None and isinstance(aircraft_families, str):
        aircraft_families = [aircraft_families]
    if origins is not None and isinstance(origins, str):
        origins = [origins]
    if destinations is not None and isinstance(destinations, str):
        destinations = [destinations]
    if brands is not None and isinstance(brands, str):
        brands = [brands]
    
    # Filter dataframe
    filtered_df = dataframe.copy()
    
    # Existing filters
    if airlines:
        filtered_df = filtered_df[filtered_df['airline_name'].isin(airlines)]
    if routes:
        filtered_df = filtered_df[filtered_df['route'].isin(routes)]
    if origins:
        filtered_df = filtered_df[filtered_df['origin'].isin(origins)]
    if destinations:
        filtered_df = filtered_df[filtered_df['destination'].isin(destinations)]
    
    # New filters
    if stops is not None and len(stops) > 0:
        # Handle 2+ stops
        if 2 in stops:
            stops_filter = list(set(stops) - {2}) + [2, 3, 4, 5, 6, 7, 8, 9, 10]  # Add more stop counts
            filtered_df = filtered_df[filtered_df['number_of_stops'].isin(stops_filter)]
        else:
            filtered_df = filtered_df[filtered_df['number_of_stops'].isin(stops)]
    
    if cabins is not None and len(cabins) > 0:
        filtered_df = filtered_df[filtered_df['cabin_class'].isin(cabins)]
    
    if redeye_only:
        try:
            filtered_df['departure_hour'] = pd.to_datetime(filtered_df['departure_time']).dt.hour
            filtered_df = filtered_df[(filtered_df['departure_hour'] >= 22) | (filtered_df['departure_hour'] < 5)]
        except:
            pass  # Skip if parsing fails
    
    if days_until_range is not None:
        filtered_df = filtered_df[
            (filtered_df['days_until_departure'] >= days_until_range[0]) & 
            (filtered_df['days_until_departure'] <= days_until_range[1])
        ]
    
    if departure_hour_range is not None:
        try:
            filtered_df['departure_hour'] = pd.to_datetime(filtered_df['departure_time']).dt.hour
            filtered_df = filtered_df[
                (filtered_df['departure_hour'] >= departure_hour_range[0]) & 
                (filtered_df['departure_hour'] <= departure_hour_range[1])
            ]
        except:
            pass  # Skip if parsing fails
    
    if price_range is not None:
        filtered_df = filtered_df[
            (filtered_df['price'] >= price_range[0]) & 
            (filtered_df['price'] <= price_range[1])
        ]
    
    if duration_minutes_range is not None:
        try:
            # Use the helper function to parse duration
            filtered_df['duration_minutes'] = filtered_df['total_duration'].apply(parse_duration_to_minutes)
            filtered_df = filtered_df[filtered_df['duration_minutes'].notna()]
            filtered_df = filtered_df[
                (filtered_df['duration_minutes'] >= duration_minutes_range[0]) & 
                (filtered_df['duration_minutes'] <= duration_minutes_range[1])
            ]
        except:
            pass  # Skip if parsing fails
    
    if seats_range is not None:
        filtered_df = filtered_df[
            (filtered_df['bookable_seats'] >= seats_range[0]) & 
            (filtered_df['bookable_seats'] <= seats_range[1])
        ]
    
    # Count all aircraft segments
    aircraft_counts = Counter()
    
    for aircraft_str in filtered_df['aircraft_type']:
        codes = [code.strip() for code in str(aircraft_str).split(',')]
        for code in codes:
            aircraft_name = aircraft_mapping.get(code, code)
            aircraft_counts[aircraft_name] += 1
    
    # Convert to DataFrame
    aircraft_df = pd.DataFrame(aircraft_counts.items(), columns=['Aircraft', 'Count'])
    
    if len(aircraft_df) == 0:
        # Return empty dataframe with proper columns
        return pd.DataFrame(columns=['Aircraft', 'Count', 'Family', 'Brand', 'Sort_Order'])
    
    # Add aircraft family and brand
    aircraft_df['Family'] = aircraft_df['Aircraft'].map(aircraft_family_mapping)
    aircraft_df['Brand'] = aircraft_df['Aircraft'].map(aircraft_brand_mapping)
    
    # Filter by aircraft family if specified
    if aircraft_families:
        aircraft_df = aircraft_df[aircraft_df['Family'].isin(aircraft_families)]
    
    # Filter by brand if specified
    if brands:
        aircraft_df = aircraft_df[aircraft_df['Brand'].isin(brands)]
    
    # Add sort order and sort by it
    aircraft_df['Sort_Order'] = aircraft_df['Aircraft'].map(aircraft_sort_order)
    aircraft_df['Sort_Order'] = aircraft_df['Sort_Order'].fillna(999)
    aircraft_df = aircraft_df.sort_values('Sort_Order').reset_index(drop=True)
    
    return aircraft_df

---
## 🎨 8. Interactive Dashboard

### Dashboard Features:
- **Basic Filters:** Airline, Route, Origin, Destination
- **Aircraft Filters:** Aircraft Family, Brand/Manufacturer
- **Flight Characteristics:** Number of stops, Cabin class, Red-eye flights
- **Advanced Filters:** Price range, Days until departure, Flight duration, Departure time, Available seats
- **Visualizations:** Interactive pie chart with hover details, Summary statistics
- **Currency Toggle:** Switch between EUR and USD with live conversion

### Dashboard URL:
After running this cell, access the dashboard at: [**http://127.0.0.1:8050/**](http://127.0.0.1:8050/)


In [12]:
# Helper function to determine if flight is red-eye
def is_redeye(departure_time_str):
    """Check if departure time is between 10pm and 5am"""
    try:
        hour = pd.to_datetime(departure_time_str).hour
        return hour >= 22 or hour < 5
    except:
        return False

# Updated Dash app
app = dash.Dash(__name__)

# Get unique values and ranges for filters
unique_airlines = ['All'] + sorted(df['airline_name'].dropna().unique().tolist())
unique_routes = ['All'] + sorted(df['route'].dropna().unique().tolist())
unique_families = ['All'] + sorted(list(set(aircraft_family_mapping.values())))
unique_origins = ['All'] + sorted(df['origin'].dropna().unique().tolist())
unique_destinations = ['All'] + sorted(df['destination'].dropna().unique().tolist())
unique_brands = ['All', 'Airbus', 'Boeing', 'Bombardier', 'Embraer']

# Update the ranges calculation section:
# Get ranges for sliders - with better error handling
try:
    min_price = float(df['price'].min())
    max_price = float(df['price'].max())
except:
    min_price = 0
    max_price = 1000

try:
    min_days = int(df['days_until_departure'].min())
    max_days = min(int(df['days_until_departure'].max()), 14)
except:
    min_days = 0
    max_days = 14

try:
    df['duration_minutes'] = df['total_duration'].apply(parse_duration_to_minutes)
    df_duration = df[df['duration_minutes'].notna()]
    min_duration_minutes = float(df_duration['duration_minutes'].min())
    max_duration_minutes = float(df_duration['duration_minutes'].max())
    # Convert to hours for display
    min_duration_hours = min_duration_minutes / 60
    max_duration_hours = max_duration_minutes / 60
except:
    min_duration_hours = 1
    max_duration_hours = 10

try:
    min_seats = int(df['bookable_seats'].min())
    max_seats = int(df['bookable_seats'].max())
except:
    min_seats = 1
    max_seats = 9


# Define the app layout
app.layout = html.Div([
    # ============================================================================
    # HEADER WITH LOGOS
    # ============================================================================
    html.Div([
        # Dickinson College - Seal + Text
        html.Div([
            html.Img(
                src='/assets/dickinson_seal.svg',
                style={
                    'height': '70px',
                    'marginRight': '15px'
                }
            ),
            html.Div([
                html.Span('Dickinson', style={
                    'fontSize': '28px',
                    'fontWeight': 'bold',
                    'color': '#c8102e',
                    'fontFamily': 'Times New Roman, serif',
                    'letterSpacing': '1px'
                }),
            ], style={'display': 'flex', 'flexDirection': 'column', 'justifyContent': 'center'})
        ], style={
            'display': 'flex',
            'alignItems': 'center',
            'marginRight': 'auto'
        }),
        
        # Title in the middle
        html.H1('Flight Aircraft Usage Dashboard', 
            style={
                'textAlign': 'center', 
                'color': '#2c3e50', 
                'margin': '0 20px',
                'flex': '1'
            }
        ),
        
        # Amadeus Logo (right)
        html.Img(
            src='/assets/amadeus_logo.svg',
            style={
                'height': '50px',
                'marginLeft': 'auto'
            }
        ),
    ], style={
        'display': 'flex',
        'alignItems': 'center',
        'justifyContent': 'space-between',
        'padding': '20px 40px',
        'backgroundColor': '#f8f9fa',
        'borderBottom': '3px solid #3498db',
        'marginBottom': '20px'
    }),
    
    html.Div([
        # Left side - Filters column (scrollable)
        html.Div([
            html.Div([
                # Basic Filters
                html.H3('Basic Filters', style={'color': '#34495e', 'borderBottom': '2px solid #3498db', 'paddingBottom': '10px'}),
                
                # Airline dropdown
                html.Div([
                    html.Label('Airline:', style={'fontWeight': 'bold', 'fontSize': 14}),
                    dcc.Dropdown(
                        id='airline-dropdown',
                        options=[{'label': airline, 'value': airline} for airline in unique_airlines],
                        value='All',
                        clearable=False
                    )
                ], style={'marginBottom': 15}),
                
                # Route dropdown
                html.Div([
                    html.Label('Route:', style={'fontWeight': 'bold', 'fontSize': 14}),
                    dcc.Dropdown(
                        id='route-dropdown',
                        options=[{'label': route, 'value': route} for route in unique_routes],
                        value='All',
                        clearable=False
                    )
                ], style={'marginBottom': 15}),
                
                # Origin dropdown
                html.Div([
                    html.Label('Origin:', style={'fontWeight': 'bold', 'fontSize': 14}),
                    dcc.Dropdown(
                        id='origin-dropdown',
                        options=[{'label': origin, 'value': origin} for origin in unique_origins],
                        value='All',
                        clearable=False
                    )
                ], style={'marginBottom': 15}),
                
                # Destination dropdown
                html.Div([
                    html.Label('Destination:', style={'fontWeight': 'bold', 'fontSize': 14}),
                    dcc.Dropdown(
                        id='destination-dropdown',
                        options=[{'label': dest, 'value': dest} for dest in unique_destinations],
                        value='All',
                        clearable=False
                    )
                ], style={'marginBottom': 25}),
                
                # Aircraft Filters
                html.H3('Aircraft Filters', style={'color': '#34495e', 'borderBottom': '2px solid #3498db', 'paddingBottom': '10px'}),
                
                # Aircraft Family dropdown
                html.Div([
                    html.Label('Aircraft Family:', style={'fontWeight': 'bold', 'fontSize': 14}),
                    dcc.Dropdown(
                        id='family-dropdown',
                        options=[{'label': family, 'value': family} for family in unique_families],
                        value='All',
                        clearable=False
                    )
                ], style={'marginBottom': 15}),
                
                # Brand dropdown
                html.Div([
                    html.Label('Aircraft Brand:', style={'fontWeight': 'bold', 'fontSize': 14}),
                    dcc.Dropdown(
                        id='brand-dropdown',
                        options=[{'label': brand, 'value': brand} for brand in unique_brands],
                        value='All',
                        clearable=False
                    )
                ], style={'marginBottom': 25}),
                
                # Flight Characteristics
                html.H3('Flight Characteristics', style={'color': '#34495e', 'borderBottom': '2px solid #3498db', 'paddingBottom': '10px'}),
                
                # Stops checkboxes
                html.Div([
                    html.Label('Number of Stops:', style={'fontWeight': 'bold', 'fontSize': 14}),
                    dcc.Checklist(
                        id='stops-checklist',
                        options=[
                            {'label': ' Nonstop (0)', 'value': 0},
                            {'label': ' 1 Stop', 'value': 1},
                            {'label': ' 2+ Stops', 'value': 2}
                        ],
                        value=[],
                        style={'marginTop': 5}
                    )
                ], style={'marginBottom': 15}),
                
                # Cabin class checkboxes
                html.Div([
                    html.Label('Cabin Class:', style={'fontWeight': 'bold', 'fontSize': 14}),
                    dcc.Checklist(
                        id='cabin-checklist',
                        options=[
                            {'label': ' Economy', 'value': 'ECONOMY'},
                            {'label': ' Premium Economy', 'value': 'PREMIUM_ECONOMY'},
                            {'label': ' Business', 'value': 'BUSINESS'},
                            {'label': ' First', 'value': 'FIRST'}
                        ],
                        value=[],
                        style={'marginTop': 5}
                    )
                ], style={'marginBottom': 15}),
                
                # Red-eye checkbox
                html.Div([
                    dcc.Checklist(
                        id='redeye-checkbox',
                        options=[{'label': ' Red-eye flights only (10pm-5am)', 'value': 'redeye'}],
                        value=[],
                        style={'fontWeight': 'bold'}
                    )
                ], style={'marginBottom': 25}),
                
                # Advanced Filters
                html.H3('Advanced Filters', style={'color': '#34495e', 'borderBottom': '2px solid #3498db', 'paddingBottom': '10px'}),
                
                # Currency toggle
                html.Div([
                    html.Label('Currency:', style={'fontWeight': 'bold', 'fontSize': 14, 'marginRight': 10}),
                    dcc.RadioItems(
                        id='currency-toggle',
                        options=[
                            {'label': ' EUR (€)', 'value': 'EUR'},
                            {'label': ' USD ($)', 'value': 'USD'}
                        ],
                        value='EUR',
                        inline=True,
                        style={'display': 'inline-block'}
                    ),
                    html.Br(),
                    html.Span(f'Rate: 1 EUR = ${EUR_TO_USD_RATE:.3f} USD', 
                              style={'fontSize': 11, 'color': '#34495e', 'fontStyle': 'italic'}),
                    html.Br(),
                    html.Span(f'Updated: {RATE_UPDATE_TIME}', 
                              style={'fontSize': 10, 'color': '#95a5a6'})
                ], style={'marginBottom': 20}),

                
                # Days until departure slider
                html.Div([
                    html.Label('Days Until Departure:', style={'fontWeight': 'bold', 'fontSize': 14}),
                    dcc.RangeSlider(
                        id='days-slider',
                        min=min_days,
                        max=max_days,
                        value=[min_days, max_days],
                        marks={i: str(i) for i in range(int(min_days), int(max_days)+1, 2)},
                        tooltip={"placement": "bottom", "always_visible": False}
                    )
                ], style={'marginBottom': 20}),
                
                # Departure time slider
                html.Div([
                    html.Label('Departure Hour:', style={'fontWeight': 'bold', 'fontSize': 14}),
                    dcc.RangeSlider(
                        id='departure-hour-slider',
                        min=0,
                        max=23,
                        value=[0, 23],
                        marks={0: '12am', 6: '6am', 12: '12pm', 18: '6pm', 23: '11pm'},
                        tooltip={"placement": "bottom", "always_visible": False}
                    )
                ], style={'marginBottom': 20}),
                
                # Price range slider
                html.Div([
                    html.Label(id='price-label', style={'fontWeight': 'bold', 'fontSize': 14}),
                    dcc.RangeSlider(
                        id='price-slider',
                        min=min_price,
                        max=max_price,
                        value=[min_price, max_price],
                        tooltip={"placement": "bottom", "always_visible": True}
                    )
                ], style={'marginBottom': 20}),
                
                # Duration slider - FIXED TO SHOW HOURS
                html.Div([
                    html.Label('Flight Duration (hours):', style={'fontWeight': 'bold', 'fontSize': 14}),
                    dcc.RangeSlider(
                        id='duration-slider',
                        min=min_duration_hours,
                        max=max_duration_hours,
                        value=[min_duration_hours, max_duration_hours],
                        step=0.25,  # 15 minute increments
                        marks={
                            int(min_duration_hours): f'{int(min_duration_hours)}h',
                            int(min_duration_hours + (max_duration_hours - min_duration_hours)/2): f'{int(min_duration_hours + (max_duration_hours - min_duration_hours)/2)}h',
                            int(max_duration_hours): f'{int(max_duration_hours)}h'
                        },
                        tooltip={"placement": "bottom", "always_visible": True}
                    )
                ], style={'marginBottom': 20}),
                
                # Bookable seats slider
                html.Div([
                    html.Label('Bookable Seats:', style={'fontWeight': 'bold', 'fontSize': 14}),
                    dcc.RangeSlider(
                        id='seats-slider',
                        min=min_seats,
                        max=max_seats,
                        value=[min_seats, max_seats],
                        marks={int(min_seats): str(int(min_seats)), int(max_seats): str(int(max_seats))},
                        tooltip={"placement": "bottom", "always_visible": True}
                    )
                ], style={'marginBottom': 20}),
                
            ], style={'overflowY': 'auto', 'height': '85vh', 'paddingRight': '10px'})
        ], style={'width': '22%', 'display': 'inline-block', 'verticalAlign': 'top', 'padding': '20px', 'backgroundColor': '#f8f9fa'}),
        
        # Right side - Chart
        html.Div([
            dcc.Graph(id='aircraft-pie-chart'),
            html.Div(id='chart-title', 
                     style={'textAlign': 'center', 'fontSize': 20, 'fontWeight': 'bold', 
                            'color': '#2c3e50', 'marginTop': 10}),
            html.Div(id='summary-stats', 
                     style={'textAlign': 'center', 'fontSize': 18, 'marginTop': 20, 'color': '#34495e'}),
        ], style={'width': '76%', 'display': 'inline-block', 'verticalAlign': 'top'})
    ]),
    # ============================================================================
    # FOOTER
    # ============================================================================
    html.Div([
        html.Hr(style={'margin': '30px 0', 'borderColor': '#bdc3c7'}),
        html.P([
            'Final Project - DATA 400, Fall 2025 | ',
            html.Span('Dickinson College', style={'fontWeight': 'bold', 'color': '#c8102e'}),  # Dickinson red
            ' | Created by ',
            html.Span('Kevin Tran & Will Barlett', style={'fontWeight': 'bold'}),
            ' | Data powered by ',
            html.Span('Amadeus API', style={'fontWeight': 'bold', 'color': '#0033a0'})
        ], style={
            'textAlign': 'center',
            'color': '#7f8c8d',
            'fontSize': '14px',
            'padding': '20px',
            'margin': '0'
        })
    ], style={
        'backgroundColor': '#f8f9fa',
        'marginTop': '30px'
    })
])

# Callback to update price slider marks AND values based on currency
@app.callback(
    [Output('price-slider', 'marks'),
     Output('price-slider', 'min'),
     Output('price-slider', 'max'),
     Output('price-slider', 'value'),
     Output('price-label', 'children')],
    [Input('currency-toggle', 'value'),
     State('price-slider', 'value')]
)
def update_price_slider_currency(currency, current_value):
    if currency == 'USD':
        # Convert everything to USD
        min_usd = min_price * EUR_TO_USD_RATE
        max_usd = max_price * EUR_TO_USD_RATE
        current_usd = [current_value[0] * EUR_TO_USD_RATE, current_value[1] * EUR_TO_USD_RATE]
        
        marks = {
            int(min_usd): f'${int(min_usd)}',
            int(max_usd): f'${int(max_usd)}'
        }
        label = 'Price Range (USD):'
        return marks, min_usd, max_usd, current_usd, label
    else:
        # EUR - use original values
        marks = {
            int(min_price): f'€{int(min_price)}',
            int(max_price): f'€{int(max_price)}'
        }
        label = 'Price Range (EUR):'
        # If currently in USD, convert back to EUR
        if current_value[0] > max_price:  # Detect if we're in USD mode
            current_eur = [current_value[0] / EUR_TO_USD_RATE, current_value[1] / EUR_TO_USD_RATE]
            return marks, min_price, max_price, current_eur, label
        return marks, min_price, max_price, current_value, label

@app.callback(
    [Output('aircraft-pie-chart', 'figure'),
     Output('chart-title', 'children'),
     Output('summary-stats', 'children'),
     Input('airline-dropdown', 'value'),
     Input('route-dropdown', 'value'),
     Input('family-dropdown', 'value'),
     Input('origin-dropdown', 'value'),
     Input('destination-dropdown', 'value'),
     Input('brand-dropdown', 'value'),
     Input('stops-checklist', 'value'),
     Input('cabin-checklist', 'value'),
     Input('redeye-checkbox', 'value'),
     Input('days-slider', 'value'),
     Input('departure-hour-slider', 'value'),
     Input('price-slider', 'value'),
     Input('duration-slider', 'value'),
     Input('seats-slider', 'value'),
     Input('currency-toggle', 'value')]  # NEW INPUT
)
def update_chart(selected_airline, selected_route, selected_family, selected_origin, selected_destination, selected_brand,
                stops_selected, cabins_selected, redeye_selected, days_range, departure_hour_range, 
                price_range, duration_range, seats_range, currency):
    
    # Convert duration from hours back to minutes for filtering
    duration_minutes_range = [duration_range[0] * 60, duration_range[1] * 60]
    
    # Convert price back to EUR if in USD mode (since data is in EUR)
    if currency == 'USD':
        price_range_eur = [price_range[0] / EUR_TO_USD_RATE, price_range[1] / EUR_TO_USD_RATE]
    else:
        price_range_eur = price_range
    
    # Handle 'All' selections
    airlines_filter = None if selected_airline == 'All' else selected_airline
    routes_filter = None if selected_route == 'All' else selected_route
    families_filter = None if selected_family == 'All' else selected_family
    origins_filter = None if selected_origin == 'All' else selected_origin
    destinations_filter = None if selected_destination == 'All' else selected_destination
    brands_filter = None if selected_brand == 'All' else selected_brand
    
    # Process new filters
    stops_filter = stops_selected if len(stops_selected) > 0 else None
    cabins_filter = cabins_selected if len(cabins_selected) > 0 else None
    redeye_filter = 'redeye' in redeye_selected
    
    # Get aircraft counts
    aircraft_data = count_aircraft(df, 
                                   airlines=airlines_filter, 
                                   routes=routes_filter, 
                                   aircraft_families=families_filter,
                                   origins=origins_filter,
                                   destinations=destinations_filter,
                                   brands=brands_filter,
                                   stops=stops_filter,
                                   cabins=cabins_filter,
                                   redeye_only=redeye_filter,
                                   days_until_range=days_range,
                                   departure_hour_range=departure_hour_range,
                                   price_range=price_range_eur,
                                   duration_minutes_range=duration_minutes_range,  # CONVERTED
                                   seats_range=seats_range)
    
    if len(aircraft_data) == 0:
        # Empty chart
        fig = go.Figure()
        fig.update_layout(
            annotations=[{
                'text': 'No data matches the selected filters',
                'xref': 'paper',
                'yref': 'paper',
                'showarrow': False,
                'font': {'size': 20}
            }],
            height=700
        )
        return fig, 'No Results', html.Div('No aircraft data found for selected filters'), price_label
    
    # Calculate percentages
    aircraft_data['Percentage'] = (aircraft_data['Count'] / aircraft_data['Count'].sum()) * 100
    
    # Identify which aircraft are >= 5%
    aircraft_above_threshold = set(aircraft_data[aircraft_data['Percentage'] >= 5]['Aircraft'].tolist())
    
    # Create custom text
    custom_text = [aircraft if aircraft in aircraft_above_threshold else '' 
                   for aircraft in aircraft_data['Aircraft']]
    
    labeled_count = len(aircraft_above_threshold)
    
    # Create pie chart
    fig = go.Figure(data=[go.Pie(
        labels=aircraft_data['Aircraft'],
        values=aircraft_data['Count'],
        hole=0.3,
        text=custom_text,
        textposition='auto',
        textinfo='text+percent',
        textfont=dict(size=14),
        hovertemplate='<b>%{label}</b><br>Family: ' + aircraft_data['Family'].astype(str) + '<br>Brand: ' + aircraft_data['Brand'].astype(str) + '<br>Count: %{value}<br>Percentage: %{percent}<extra></extra>',
        marker=dict(line=dict(color='white', width=2)),
        sort=False
    )])
    
    # Build title
    title_text = f'Aircraft Usage (Labels ≥5%)'
    
    fig.update_layout(
        showlegend=True,
        legend=dict(
            orientation="v",
            yanchor="middle",
            y=0.5,
            xanchor="left",
            x=1.02,
            font=dict(size=14),
            itemsizing='constant',
            traceorder='normal'
        ),
        height=700,
        margin=dict(l=20, r=250, t=20, b=20)
    )
    
    # Summary statistics - with currency conversion
    total_segments = aircraft_data['Count'].sum()
    num_aircraft_types = len(aircraft_data)
    
    # Calculate average price from filtered data (you can add this later if needed)
    # For now just show the summary without price
    
    summary = html.Div([
        html.Span(f'Total Aircraft Segments: {total_segments:,}', 
                  style={'marginRight': 40, 'fontWeight': 'bold'}),
        html.Span(f'Aircraft Types: {num_aircraft_types}',
                  style={'marginRight': 40, 'fontWeight': 'bold'}),
        html.Span(f'Labeled: {labeled_count} types',
                  style={'fontWeight': 'bold', 'color': '#e74c3c'})
    ])
    
    return fig, title_text, summary
# Run the app
if __name__ == '__main__':
    app.run(debug=True, port=8050)


---
## 📝 Notes & Credits

**Project Information:**
- Final Project for DATA 400 (Fall 2025)
- Dickinson College
- Created by Kevin Tran & Will Barlett

**Data Source:**
- Amadeus Flight API
- Collection Period: November 10 - December 19, 2025

**Technologies Used:**
- Python 3.13
- Dash & Plotly for interactive visualization
- Pandas for data processing
- Dropbox API for cloud storage

---
*Dashboard is best viewed in Chrome or Firefox at 1920x1080 resolution or higher.*
